# LELA60341: Research Methods in Computational and Corpus Linguistics Week 8

## String algorithms

An **algorithm** is a step-by-step method for solving a problem, precise enough for a computer to carry out. This workbook looks at algorithms that work on **strings** (text): how Python stores characters, a simple way of writing secret messages (the Caesar cipher), and the question behind every `"word" in text`: how do you find a pattern in a text? It uses the Python from the earlier LELA60341 workbooks, including the classes from Week 7.

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 1 · Characters and strings

Inside the computer, each character is stored as a number. **Unicode** is the international standard that gives every character of every writing system its own number, called its **code point**: `a` is 97, `é` is 233 and `€` is 8364. `ord(c)` turns a character into its number, and `chr(n)` turns a number back into a character. The capital letters have **consecutive** numbers, from `'A'` (65) to `'Z'` (90), so `ord(c) - ord('A')` turns a capital letter into a position from 0 to 25.

Strings are **immutable** (Week 2), so to "edit" one, turn it into a list of characters with `list(s)`, change the list, then glue it back together with `"".join(...)`. Joining a list once is also much faster than adding characters to a string one at a time with `+=`, which copies the whole string each time.

In [ ]:
print(ord("A"), ord("Z"), chr(65), [ord(c) - ord("A") for c in "ABCXYZ"])

chars = list("bird")
chars[0] = "w"
print("".join(chars))

document = "It's 9:30, Mr. O'Brien!"
print("".join([c for c in document if c.isalpha()]))     # isalpha(): is c a letter?

---
# Part 2 · The Caesar cipher

A **cipher** is a method for disguising a message so that only someone who knows the secret can read it. Turning the readable message (the **plaintext**) into the disguised one (the **ciphertext**) is called **encryption**, and turning it back is **decryption**. The study of such methods is **cryptography**.

One of the oldest ciphers is named after Julius Caesar, who is said to have used it for military messages. The **Caesar cipher** replaces each letter with the letter a fixed number of places later in the alphabet. With a shift of 3, A → D, B → E, and so on, and at the end the alphabet **wraps around**: X → A, Y → B, Z → C. If letters are numbered 0–25, a shift of *r* sends letter *i* to `(i + r) % 26`. Decryption uses `(i - r) % 26`. With `% 26`, Python always gives an answer from 0 to 25, even for negative numbers: `(0 - 3) % 26` is 23.

Here is a `CaesarCipher` class, which uses what you learned in Week 7:

* The constructor works out, once, a 26-character string saying what each letter becomes when encrypting (`_forward`) and when decrypting (`_backward`).
* `encrypt` and `decrypt` both hand the work to an internal helper method, `_transform`. It replaces each capital letter `c` with `code[ord(c) - ord("A")]`. `isupper()` asks whether a character is a capital letter.

In [ ]:
class CaesarCipher:
    """Encrypt and decrypt messages with a Caesar cipher."""

    def __init__(self, shift):
        encoder = [None] * 26                     # a list of 26 empty slots
        decoder = [None] * 26
        for k in range(26):
            encoder[k] = chr((k + shift) % 26 + ord("A"))
            decoder[k] = chr((k - shift) % 26 + ord("A"))
        self._forward = "".join(encoder)
        self._backward = "".join(decoder)

    def encrypt(self, message):
        return self._transform(message, self._forward)

    def decrypt(self, secret):
        return self._transform(secret, self._backward)

    def _transform(self, original, code):
        msg = list(original)
        for k in range(len(msg)):
            if msg[k].isupper():
                j = ord(msg[k]) - ord("A")        # a position from 0 to 25
                msg[k] = code[j]
        return "".join(msg)

cipher = CaesarCipher(3)
print(cipher._forward)
coded = cipher.encrypt("THE EAGLE IS IN PLAY; MEET AT JOE'S.")
print("Secret: ", coded)
print("Message:", cipher.decrypt(coded))

### 🏋️ Exercise 1 · Practice — Crack it by brute force

There are only 26 possible shifts, so an eavesdropper can simply **try them all**. This is a **brute-force** algorithm: try every possibility and keep the best one.

Complete `crack(secret)`. For every shift from 0 to 25 it decrypts `secret` and gives the result a **score**: the number of its words that are in `COMMON_WORDS` (the helper `score` is done for you). It returns the shift and plaintext with the **highest** score.

In [ ]:
COMMON_WORDS = {"THE", "OF", "AND", "TO", "A", "IN", "IS", "IT", "THAT", "WAS",
                "FOR", "ON", "ARE", "WITH", "AS", "AT", "BE", "THIS", "HAVE", "FROM"}

secret = "FVB ZOHSS RUVD H DVYK IF AOL JVTWHUF PA RLLWZ. AOHA PZ AOL PKLH HA AOL OLHYA VM AOPZ MPLSK."

def score(text):
    return len([w for w in text.split() if w.strip(".,;!?") in COMMON_WORDS])

def crack(secret):
    best_shift = 0
    best_text = secret
    best_score = -1
    for shift in range(26):
        candidate = ???               # decrypt secret with this shift
        if ???:                       # better than the best so far?
            best_shift = shift
            best_text = candidate
            best_score = score(candidate)
    return best_shift, best_text

print(crack(secret))

In [ ]:
# ✅ Check
check("finds the shift", lambda: crack(secret)[0] == 7)
check("finds the plaintext", lambda: crack(secret)[1].startswith("YOU SHALL KNOW A WORD BY THE COMPANY IT KEEPS."))

The message is a famous saying of the linguist J. R. Firth (1957). It sums up **distributional semantics**: the idea that we can learn what a word means from the words that tend to occur around it, which underlies much of modern computational linguistics.

---
# Part 3 · Substrings, prefixes and suffixes

Here is some useful vocabulary for talking about strings. For a string `S`:

* `S[j:k]` is the **substring** from position `j` up to **but not including** position `k` (Week 2's slices).
* A **prefix** is a substring that starts at the beginning: `S[:k]`. A **suffix** is a substring that runs to the end: `S[j:]`.
* The empty string `""` is a prefix and a suffix of every string, and so is `S` itself.

In [ ]:
S = "linguistics"
print([S[:k] for k in range(len(S) + 1)])      # every prefix
print([S[j:] for j in range(len(S) + 1)])      # every suffix
print(S.startswith("ling"), S.endswith("ics"))

---
# Part 4 · Finding a pattern: brute force

We are given a **text** `T` and a **pattern** `P` of length *m*. Does `P` occur in `T`? If so, we want the **lowest position** `i` where it starts, so that `T[i:i+m] == P`, and otherwise `-1`. This is what `T.find(P)` does, and `in`, `count`, `replace` and `split` all have to solve the same problem.

The **brute-force** approach tries every starting position `i`, and compares `P` with `T` character by character, stopping at the first mismatch. On the small example below, each row below is one value of `i`. Matching characters are in capitals, and `x` marks the mismatch that ends the row:

```
T:  a b a c a a b a d c a b a c a b a a b b
i=0 A B A C A x                              (5 matches, then a mismatch)
i=1   x                                      (a mismatch straight away)
i=2     A x
...
i=10                    A B A C A B          ✔ match found at 10
```

### 🏋️ Exercise 2 · Practice — Count the work

Complete `find_brute_count(T, P)`, which returns **two** values: the position (or `-1`), and the number of times a character of `T` was compared with a character of `P`. For the example above the answer is **27** comparisons.

In [ ]:
def find_brute_count(T, P):
    n = len(T)
    m = len(P)
    comparisons = 0
    for i in range(n - m + 1):              # try every possible starting position in T
        k = 0                               # a position in P
        while k < m:
            ???                   # one more comparison is about to happen
            if T[i + k] != P[k]:
                break
            k += 1
        if k == m:                          # we reached the end of the pattern
            return ???
    return -1, comparisons

print(find_brute_count("abacaabadcabacabaabb", "abacab"))

In [ ]:
# ✅ Check
check("example: position 10, 27 comparisons", lambda: find_brute_count("abacaabadcabacabaabb", "abacab") == (10, 27))
check("no match: every position tried", lambda: find_brute_count("aaaa", "ab") == (-1, 6))

### 🏋️ Exercise 3 · Practice — A concordance

In corpus work we usually want **every** match, for example to build a **concordance**: each occurrence of a word with some context either side.

1. Complete `find_all(T, P)`, which returns a list of **all** the starting positions of `P` in `T`. Try every `i` and keep those where the slice `T[i:i+len(P)]` equals `P`.
2. Complete `concordance`, which returns one line per match: `width` characters before the match, the match in square brackets, and `width` characters after it. The left context is done for you.

In [ ]:
def find_all(T, P):
    return ???

def concordance(T, word, width=10):
    lines = []
    for i in find_all(T, word):
        left = T[max(0, i - width):i]                          # (done for you)
        right = ???                                           # the width characters after the match
        lines.append(left + "[" + word + "]" + right)
    return lines

print(find_all("banana", "ana"))
text = "she sells sea shells by the sea shore; the shells she sells are sea shells"
for line in concordance(text, "sea", 8):
    print(line)

In [ ]:
# ✅ Check
check("overlapping matches", lambda: find_all("banana", "ana") == [1, 3])
check("concordance", lambda: concordance(text, "sea", 8)[0] == "e sells [sea] shells " and len(concordance(text, "sea", 8)) == 3)

---
# Part 5 · Boyer–Moore: skipping ahead

Does a pattern matcher have to look at **every** character of the text? No! The **Boyer–Moore** algorithm, named after Robert Boyer and J Strother Moore who published it in 1977, uses two ideas (we use a simplified version here):

1. **Compare from the end of the pattern backwards.**
2. **Jump on a mismatch.** When a text character `c` doesn't match: if `c` **doesn't occur anywhere in the pattern**, slide the pattern **completely past** it. Otherwise, slide the pattern so that the **last occurrence of `c` in the pattern** lines up with it.

```
Text:     . . . . e . . . . s . . . . . . . . . .
Pattern:  s u s h i                     'e' is not in "sushi": jump right past it
                    s u s h i           's' IS in "sushi": line up its last 's'
                        s u s h i
```

To make the jumps quick, Boyer–Moore first builds a dict `last`, mapping each character of the pattern to the position of its **last** occurrence. `last.get(c, -1)` gives −1 for characters that aren't in the pattern. Here is a Boyer–Moore function, with a comparison counter added. The line `i += m - min(k, j + 1)` is the jump rule above, written as arithmetic.

In [ ]:
def find_boyer_moore_count(T, P):
    n = len(T)
    m = len(P)
    last = {}
    for k in range(m):
        last[P[k]] = k                 # a later occurrence overwrites an earlier one
    i = m - 1                          # a position in T
    k = m - 1                          # a position in P
    comparisons = 0
    while i < n:
        comparisons += 1
        if T[i] == P[k]:               # a matching character
            if k == 0:
                return i, comparisons  # the pattern begins at position i of the text
            i -= 1                     # look at the previous character
            k -= 1                     # of both T and P
        else:
            j = last.get(T[i], -1)     # -1 if T[i] is not in the pattern
            i += m - min(k, j + 1)     # jump
            k = m - 1                  # restart at the end of the pattern
    return -1, comparisons

print(find_boyer_moore_count("abacaabadcabacabaabb", "abacab"))   # 13 comparisons, against 27 for brute force

🔮 **Predict:** on ordinary English text, will Boyer–Moore make more or fewer comparisons than brute force? Will the difference grow or shrink as the pattern gets **longer**?

In [ ]:
english = ("It is a truth universally acknowledged, that a single man in possession "
           "of a good fortune, must be in want of a wife. ") * 50
english += "The quick brown fox jumps over the lazy dog."

for P in ["dog", "lazy dog", "quick brown fox jumps"]:
    bf = find_brute_count(english, P)[1]
    bm = find_boyer_moore_count(english, P)[1]
    print(f"{P!r:25} brute force: {bf:6}   Boyer–Moore: {bm:5}   (text length {len(english)})")

On English text, Boyer–Moore looks at only a **fraction** of the characters, and **longer patterns allow longer jumps**.

---
## ✏️ Summary

You can now convert between characters and numbers; use, and crack by brute force, a Caesar cipher written as a class; talk about substrings, prefixes and suffixes; write a brute-force pattern matcher and use it for a concordance; and explain how Boyer–Moore skips ahead. Next week: **Huffman coding**, which compresses text using the idea of a **prefix**.

*These materials are adapted from M. T. Goodrich, R. Tamassia and M. H. Goldwasser, *Data Structures and Algorithms in Python* (Wiley, 2013).*